# Orquestador multi-agente — demostración del flujo de delegación

Este notebook muestra el orquestador resolviendo una consulta que **ningún
agente puede responder solo**:

> *Nuestros servicios asíncronos se vienen cayendo seguido. ¿Qué dice nuestra
> documentación técnica sobre las causas más probables, y qué muestran nuestros
> propios incidentes del último año?*

La primera mitad necesita búsqueda semántica sobre la documentación del equipo.
La segunda necesita consultar la bitácora de incidentes y calcular estadísticas
sobre ella. Son dos dominios distintos, y el informe final tiene sentido solo si
alguien los conecta.

Lo que se va a ver, en orden:

1. La topología del grafo.
2. Las herramientas acotadas de cada especialista.
3. **El flujo de delegación paso a paso**, en vivo.
4. La trazabilidad: qué agente aportó qué.
5. La rúbrica de validación y el informe final.
6. Los tests con LLM simulado, que cubren lo mismo sin gastar API.

In [1]:
import sys
from pathlib import Path

# El notebook corre desde pre-entrega-6/, igual que el CLI.
sys.path.insert(0, str(Path.cwd()))

from config import MAX_PASOS_SUPERVISOR, RECURSION_LIMIT, get_model_name, get_provider, setup_logging
from graph import abrir_orquestador, config_thread, diagrama_mermaid, entrada
from seed_db import asegurar_db

# Los logs del orquestador van a WARNING: en el notebook queremos ver el flujo
# de delegación, no la telemetría de cada llamada. El CLI (`python main.py
# --paso-a-paso`) los muestra en INFO.
import logging
setup_logging(logging.WARNING)
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

asegurar_db()

print(f"proveedor:            {get_provider()} / {get_model_name()}")
print(f"tope del supervisor:  {MAX_PASOS_SUPERVISOR} pasos")
print(f"recursion_limit:      {RECURSION_LIMIT}")

proveedor:            gemini / gemini-flash-lite-latest
tope del supervisor:  6 pasos
recursion_limit:      25


## 1. La topología

Jerárquica: un supervisor enruta, dos especialistas trabajan, un validador
verifica y un sintetizador cierra. Los especialistas **nunca** deciden quién
sigue — siempre devuelven el control al supervisor.

El diagrama se genera desde el grafo compilado, no se escribe a mano.

In [2]:
print(diagrama_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	validador(validador)
	sintetizador(sintetizador)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	investigador --> supervisor;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	supervisor -.-> validador;
	validador -.-> sintetizador;
	validador -.-> supervisor;
	sintetizador --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Las herramientas de cada especialista

Acotadas a propósito. El investigador no puede calcular y el analista no puede
buscar: esa frontera es lo que hace que el supervisor tenga algo que coordinar.

Las tres herramientas del analista reciben los datos por `InjectedState`, así
que el LLM decide *cuándo* llamarlas pero no le dicta los registros.

In [3]:
from agents.analyst_agent import HERRAMIENTAS as TOOLS_ANALISTA
from agents.research_agent import HERRAMIENTAS as TOOLS_INVESTIGADOR

for nombre, herramientas in [("INVESTIGADOR", TOOLS_INVESTIGADOR), ("ANALISTA", TOOLS_ANALISTA)]:
    print(f"\n{nombre}")
    for h in herramientas:
        argumentos = [a for a in h.args.keys()]
        primera_linea = (h.description or "").strip().split("\n")[0]
        print(f"  - {h.name}({', '.join(argumentos) or '— lee del estado —'})")
        print(f"      {primera_linea}")


INVESTIGADOR
  - buscar_en_documentacion(consulta)
      Busca en la documentación técnica del equipo sobre programación asíncrona.
  - listar_incidentes(meses, categoria)
      Trae los incidentes de producción del equipo desde la bitácora interna.

ANALISTA
  - resumir_incidentes(— lee del estado —)
      Calcula las métricas y las agrupaciones de los incidentes recuperados.
  - detectar_atipicos(umbral_iqr)
      Encuentra los incidentes de duración anormalmente alta.
  - validar_registros(— lee del estado —)
      Verifica que los incidentes recuperados cumplan el esquema esperado.


## 3. El flujo de delegación, en vivo

`astream(stream_mode="updates")` entrega lo que devolvió cada nodo a medida que
termina. Es la forma de ver la delegación mientras pasa, en lugar de esperar al
resultado final.

Prestá atención al orden: el supervisor manda primero al investigador (sin datos
no hay nada que analizar), después al analista, y recién entonces al validador.

In [4]:
from main import CONSULTA_DEMO, describir_paso

# Checkpointer en memoria: la demo no deja estado de corridas anteriores.
async with abrir_orquestador(":memory:") as orquestador:
    config = config_thread("demo-notebook")
    async for fragmento in orquestador.astream(
        entrada(CONSULTA_DEMO), config=config, stream_mode="updates"
    ):
        for nodo, actualizacion in fragmento.items():
            print(f"\n> {describir_paso(nodo, actualizacion)}")

    final = (await orquestador.aget_state(config)).values


> SUPERVISOR -> investigador
    Faltan cuatro de cuatro condiciones en la rúbrica ya que ningún agente ha trabajado todaví



> INVESTIGADOR aportó 2 hallazgo(s):
    [buscar_en_documentacion] 4 fragmentos de documentación (vía pinecone) sobre: El event loop de asyncio, Errores comunes en asyncio, Rendimiento y depuración
    [listar_incidentes] 20 incidentes entre 2025-10-06 y 2026-10-06, en 5 categorías



> SUPERVISOR -> analista
    Ya se cuenta con la evidencia documental y los datos cuantitativos recolectados por el inv



> ANALISTA aportó 3 hallazgo(s):
    [resumir_incidentes] 20 incidentes, 1225 minutos caídos en total; mediana 37.5 min; causa dominante bloqueo-event-loop (45.0%)
    [detectar_atipicos] 1 atípico(s); el mayor es INC-0214 con 480 min (12.8x la mediana)
    [validar_registros] 19/20 registros válidos (95.0%), 1 con problemas



> SUPERVISOR -> validador
    Las cuatro condiciones de la rúbrica están cumplidas, por lo que el trabajo está listo par

> VALIDADOR -> APROBADO
    Las cuatro condiciones de la rúbrica se cumplen.



> SINTETIZADOR -> informe final redactado


## 4. Trazabilidad: qué agente aportó qué

El estado no guarda solo la conversación: guarda `hallazgos`, una lista de
contribuciones estructuradas con su procedencia. Cada número del informe final
se puede rastrear hasta la herramienta que lo produjo.

In [5]:
print(f"{'#':>2}  {'agente':<14} {'herramienta':<24} resumen")
print("-" * 100)
for i, h in enumerate(final["hallazgos"], start=1):
    print(f"{i:>2}  {h.agente:<14} {h.herramienta:<24} {h.resumen}")

print(f"\npasos de supervisión: {final['pasos_del_supervisor']}")
print(f"hallazgos totales:    {len(final['hallazgos'])}")

 #  agente         herramienta              resumen
----------------------------------------------------------------------------------------------------
 1  investigador   buscar_en_documentacion  4 fragmentos de documentación (vía pinecone) sobre: El event loop de asyncio, Errores comunes en asyncio, Rendimiento y depuración
 2  investigador   listar_incidentes        20 incidentes entre 2025-10-06 y 2026-10-06, en 5 categorías
 3  analista       resumir_incidentes       20 incidentes, 1225 minutos caídos en total; mediana 37.5 min; causa dominante bloqueo-event-loop (45.0%)
 4  analista       detectar_atipicos        1 atípico(s); el mayor es INC-0214 con 480 min (12.8x la mediana)
 5  analista       validar_registros        19/20 registros válidos (95.0%), 1 con problemas

pasos de supervisión: 3
hallazgos totales:    5


### Los datos crudos siguen ahí

Los payloads se guardan tal como los devolvieron las herramientas. El
sintetizador lee de acá —no del historial de mensajes— y por eso los números del
informe son los de la base y no una reconstrucción del modelo.

In [6]:
import json

resumen = next(h for h in final["hallazgos"] if h.herramienta == "resumir_incidentes")
print(json.dumps(resumen.datos["metricas_minutos"], ensure_ascii=False, indent=2))
print("\nPor categoría:")
for fila in resumen.datos["por_categoria"]:
    print(f"  {fila['categoria']:<26} {fila['cantidad']:>2} incidentes "
          f"({fila['porcentaje']:>5}%)  {fila['minutos_totales']:>5} min")

atipicos = next(h for h in final["hallazgos"] if h.herramienta == "detectar_atipicos")
print(f"\nAtípicos (IQR, límite {atipicos.datos['limite_superior']} min):")
for a in atipicos.datos["atipicos"]:
    print(f"  {a['incidente_id']}  {a['minutos_caidos']} min  "
          f"({a['veces_la_mediana']}x la mediana)")

{
  "media": 61.25,
  "mediana": 37.5,
  "minimo": -5,
  "maximo": 480,
  "desvio_estandar": 101.68
}

Por categoría:
  bloqueo-event-loop          9 incidentes ( 45.0%)    453 min
  timeout-no-configurado      4 incidentes ( 20.0%)    152 min
  cancelacion-mal-manejada    3 incidentes ( 15.0%)    560 min
  tarea-huerfana              2 incidentes ( 10.0%)     30 min
  fuga-de-tareas              2 incidentes ( 10.0%)     30 min

Atípicos (IQR, límite 105.0 min):
  INC-0214  480 min  (12.8x la mediana)


## 5. La rúbrica y la validación

El validador **no usa LLM**. Recorre los hallazgos y comprueba qué herramientas
corrieron efectivamente. Si el supervisor declarara el trabajo terminado sin
haber pasado por el analista, el validador lo detectaría y lo devolvería: no le
cree al modelo, le cree al estado.

In [7]:
rubrica = final["rubrica_actual"]

for criterio, cumplido in rubrica.model_dump().items():
    print(f"  [{'x' if cumplido else ' '}] {criterio}")

print(f"\nrúbrica completa: {rubrica.completa}")
print(f"validación:       {final['motivo_validacion']}")

  [x] hay_evidencia_documental
  [x] hay_datos_cuantitativos
  [x] hay_analisis_estadistico
  [x] hay_validacion_de_datos

rúbrica completa: True
validación:       Las cuatro condiciones de la rúbrica se cumplen.


## 6. El informe final

Lo redacta el sintetizador a partir de los hallazgos estructurados. Conecta las
dos mitades: lo que dice la documentación sobre la causa y lo que muestran
nuestros propios datos.

In [8]:
print(final["informe_final"])

La documentación técnica advierte que el uso de llamadas sincrónicas pesadas o procesamiento intensivo dentro de corrutinas congela el event loop de asyncio, afectando todo el programa. Nuestros datos confirman que este es el principal problema operativo, acumulando el 45.0% de los incidentes. Además, se detectó un registro con problemas (INC-0231 con -5 minutos caídos), lo que afecta la confianza en las métricas.

En el último año registramos 20 incidentes y 1225 minutos caídos en total. La causa dominante es el `bloqueo-event-loop`, con 9 incidentes (45.0%) y 453 minutos totales, impulsado por prácticas documentadas como el uso de `requests.get()`, `time.sleep()`, lectura de archivos grandes y serialización de JSONs enormes dentro del loop. Le siguen `timeout-no-configurado` con 4 incidentes (20.0% y 152 minutos), `cancelacion-mal-manejada` con 3 incidentes (15.0% y 560 minutos), `tarea-huerfana` con 2 incidentes (10.0% y 30 minutos) y `fuga-de-tareas` con 2 incidentes (10.0% y 30 mi

## 7. Lo mismo, sin gastar API: los tests con LLM simulado

Todo lo anterior cuesta llamadas al modelo y depende de que hoy decida lo mismo
que ayer. Sirve como demostración, no como test.

La suite reemplaza el modelo por una cola de respuestas prefabricadas
(`tests/stub_llm.py`) y ejercita el grafo entero de forma determinista: el ruteo,
los ciclos ReAct, el tope de pasos, el paso de contexto y la persistencia. Corre
sin red y sin ninguna API key.

In [9]:
import subprocess

resultado = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q", "--no-header", "-p", "no:warnings"],
    capture_output=True, text=True, cwd=str(Path.cwd()),
)
print(resultado.stdout.strip())
print("exit code:", resultado.returncode)

..........................                                               [100%]
26 passed in 0.71s
exit code: 0


---

### Resumen de lo demostrado

| Requisito de la consigna | Dónde se vio |
|---|---|
| Topología jerárquica con supervisor como router | celdas 1 y 3 |
| Dos especialistas con dominios distintos | celda 2 |
| Estado compartido estructurado con trazabilidad | celda 4 |
| El supervisor decide si está completo o hay que refinar | celda 5 |
| Validación antes del `END` | celda 5 |
| Freno contra el supervisor infinito | `MAX_PASOS_SUPERVISOR`, celdas 1 y 7 |
| Contaminación de contexto evitada | `InjectedState`, celdas 2 y 4 |